In [3]:
def write_as_PDB(final_coords, output_file_name):   # add additional args as needed
    '''
    :final_coords: expects z,y,x
    '''    

    with open(output_file_name, "w") as text_file:
        for i in range(final_coords.shape[0]):
            print(f"ATOM  {str(i+1):>5} {' C':<4}{' '}{'HIS':>3} {'A'}{'1':>4}{' '}   {final_coords[i][2]:>8}{final_coords[i][1]:>8}{final_coords[i][0]:>8}{'1.00':>6}{'16.00':>6}{'':6}{' ':<4}{'C':>2}", file=text_file)

## Usage

About:
- This is for testing various clustering methods.
- **Input**: Unet predictions as .mrc
- **Output**: clustered coordinates (cluster centers) as a .pdb

To use this cell:
1) Toggle the **flags** at the top (flags for which models to run and a flag for filtering outliers). 
2) Change the **threshold** if desired. The generated files will automatically have the new threshold appended to their file name. 
3) To change the input mrc file, just change the variable called "FILE". For now, need to manually set num_atoms (see note 3).
4) To change the output file names, manually go thru the code and edit each line that starts with "write_as_PDB..."

- **note 1**: User needs to manually create a directory called "clustering_results" or will get an error since I didn't use os or pathlib (*will prob add that next*).
    - **OR** just drop the directory from the file names (i.e. clustering_results/GMM_clusters_threshold_0_5.pdb --> GMM_clusters_threshold_0_5.pdb). Can name files wtv you want
    - **OR** just name the output files whatever you want.
- **note 2**: Change the FILE variable to whatever mrc you want to use as input. It's currently set to one of my local files, so will get an error if left unchanged.
- **note 3**: Still working on calculating num_atoms, so you have to manually set that for now. If you get an error, it's prob set too high.
- **note 4**: BOTH IsolationForest and most of the sklearn clustering methods have randomstate. I've set them to 0 (may have missed one) for reproducibility and debugging.
- **note 5**: This notebook doesn't use mrc headers (since just a test notebook); it assumes coordinates are Z,Y,X.

In [26]:
import mrcfile
import numpy as np
from sklearn.ensemble import IsolationForest
from timeit import default_timer as timer    # for measuring execution time

# Toggle which clustering algos to run
MEANSHIFTFLAG = True
KMEANSFLAG = False
DBSCANFLAG = False
OPTICSFLAG = False
GMMFLAG = True
AGGLOMERATIVEFLAG = False

# Toggle whether to filter outliers or not, uses sklearn IsolationForest
FILTEROUTLIERSFLAG = True

# Threshold
threshold = 0.51     # the threshold below uses >=, so set slighly above 0.5, could just change it to > instead of >=

# Read MRC data
#FILE = r"c:\Users\noelu\Downloads\Testing clustering\emd_0128.map_key_atoms_2_possibility.mrc"
#FILE = r"C:\Users\noelu\BE 177A Capstone\CryoREAD\zzzz_SANDBOX\cryoRead_files\4_26_24\21051_sugar_ring_new_1_possibility.mrc"
FILE = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065.map_key_atoms_2_possibility.mrc"
#FILE = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065.map_key_atoms_3_possibility.mrc"
#TODO: calculate num_atoms instead of explicitly setting it
#num_atoms = 4
#num_atoms = 170
num_atoms = 50

with mrcfile.open(FILE) as mrc:
    density = mrc.data
    density = np.squeeze(density)

# Convert 3D densities to 2D array of shape (n, 3)
coordinates = np.argwhere(density >= threshold)      # returns the indices of densities that are above threshold, NOTE: should maybe change to exclusive greater than
print("total", coordinates.shape)

if FILTEROUTLIERSFLAG == True:
    # Apply outlier detection
    outlier_detector = IsolationForest(contamination='auto', random_state=0)  # Adjust contamination parameter as needed
    outlier_labels = outlier_detector.fit_predict(coordinates)
    outliers = coordinates[outlier_labels == -1]
    inliers = coordinates[outlier_labels == 1]
    print("inliers", inliers.shape)
    print("outliers", outliers.shape)
    print("=== Summary ===")
    # these 2 PDBs are only generated when FILTEROUTLIERSFLAG is True
    write_as_PDB(inliers, "clustering_results/iso_forest_inliers" + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')
    write_as_PDB(outliers, "clustering_results/iso_forest_outliers" + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')

    coordinates = inliers
else:
    # this PDB is only generated when FILTEROUTLIERSFLAG is False
    write_as_PDB(coordinates, "clustering_results/iso_forest_all" + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if DBSCANFLAG == True:
    from sklearn.cluster import DBSCAN
    start = timer()
    # Apply DBSCAN
    dbscan = DBSCAN(eps=2, min_samples=10)  # Adjust parameters as needed
    dbscan.fit(coordinates)
    labels = dbscan.labels_

    # Extract cluster centers
    unique_labels = set(labels) - {-1}  # Exclude noise points
    cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
    cluster_centers = np.round(cluster_centers, 3)
    print("DBSCAN", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"DBSCAN execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/dbscan_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if AGGLOMERATIVEFLAG == True:
    from sklearn.cluster import AgglomerativeClustering
    start = timer()
    # Apply Agglomerative Clustering
    agglo = AgglomerativeClustering(n_clusters=num_atoms, linkage='ward')  # Adjust number of clusters and linkage as needed
    labels = agglo.fit_predict(coordinates)
    unique_labels = np.unique(labels)

    # Extract cluster centers
    cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
    cluster_centers = np.round(cluster_centers, 3)
    print("Agglomerative", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"Agglomerative execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/agglomerative_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if GMMFLAG == True:
    from sklearn.mixture import GaussianMixture
    start = timer()
    # Apply Gaussian Mixture Models
    gmm = GaussianMixture(n_components=num_atoms, covariance_type='full', random_state=0)  # Adjust number of components as needed
    gmm.fit(coordinates)
    cluster_centers = gmm.means_
    cluster_centers = np.round(cluster_centers, 3)
    print("GMM", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"GMM execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/gmm_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if KMEANSFLAG == True:
    from sklearn.cluster import KMeans
    start = timer()
    # Apply K-means
    kmeans = KMeans(n_clusters=num_atoms, random_state=0)  # Adjust number of clusters as needed
    kmeans.fit(coordinates)
    cluster_centers = kmeans.cluster_centers_
    cluster_centers = np.round(cluster_centers, 3)
    print("Kmeans", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"Kmeans execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/kmeans_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if MEANSHIFTFLAG == True:
    from sklearn.cluster import MeanShift
    start = timer()
    # Apply Mean Shift
    meanshift = MeanShift(bandwidth=3)  # Adjust bandwidth as needed
    meanshift.fit(coordinates)
    cluster_centers = meanshift.cluster_centers_
    cluster_centers = np.round(cluster_centers, 3)
    print("Meanshift", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"Meanshift execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/meanshift_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if OPTICSFLAG == True:
    from sklearn.cluster import OPTICS
    start = timer()
    # Apply DBSCAN
    optics = OPTICS(min_samples=2, max_eps=1.1)  # Adjust parameters as needed
    optics.fit(coordinates)
    labels = optics.labels_

    # Extract cluster centers
    unique_labels = set(labels) - {-1}  # Exclude noise points
    cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
    cluster_centers = np.round(cluster_centers, 3)

    print("OPTICS", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"OPTICS execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/optics_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')

total (1163, 3)
GMM (50, 3)
GMM execution time: 0.07366829994134605s
Meanshift (75, 3)
Meanshift execution time: 0.7622742000967264s
